<a href="https://colab.research.google.com/github/192565027simats/CSA6102/blob/main/EXP39-Cloud%20Audit%20Log%20Anomalous%20Download%20Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime
from collections import Counter

LOG_FMT = "%Y-%m-%d %H:%M:%S"


def build_baseline_ips(logs):
    """
    Return a dictionary mapping each user to their
    most frequently used IP address.
    """
    by_user = {}

    for entry in logs:
        by_user.setdefault(entry["user"], []).append(entry["ip"])

    return {
        user: Counter(ips).most_common(1)[0][0]
        for user, ips in by_user.items()
    }


def flag_anomalous_downloads(logs, business_start=8, business_end=20):
    """
    Flag download events that:
    1. Come from an IP different from the user's baseline.
    2. Occur outside business hours.
    """

    baseline = build_baseline_ips(logs)
    flagged = []

    for entry in logs:

        if entry["action"] != "download":
            continue

        ts = datetime.strptime(entry["timestamp"], LOG_FMT)
        reasons = []

        if entry["ip"] != baseline.get(entry["user"]):
            reasons.append("IP differs from user baseline")

        if not (business_start <= ts.hour < business_end):
            reasons.append("Outside business hours")

        if reasons:
            flagged.append({
                **entry,
                "reasons": reasons,
            })

    return flagged


# ---------------- Sample Input ----------------

logs = [
    {
        "user": "alice",
        "action": "view",
        "file": "roadmap.docx",
        "timestamp": "2026-03-01 10:00:00",
        "ip": "10.0.0.5",
    },
    {
        "user": "alice",
        "action": "view",
        "file": "budget.xlsx",
        "timestamp": "2026-03-02 11:00:00",
        "ip": "10.0.0.5",
    },
    {
        "user": "alice",
        "action": "download",
        "file": "report.pdf",
        "timestamp": "2026-03-03 14:00:00",
        "ip": "10.0.0.5",
    },
    {
        "user": "alice",
        "action": "download",
        "file": "customer_database.csv",
        "timestamp": "2026-03-05 02:15:00",
        "ip": "185.220.101.7",
    },
]


# ---------------- Run Detection ----------------

flagged = flag_anomalous_downloads(logs)


# ---------------- Verification ----------------

assert len(flagged) == 1
assert flagged[0]["file"] == "customer_database.csv"
assert "IP differs from user baseline" in flagged[0]["reasons"]
assert "Outside business hours" in flagged[0]["reasons"]

print("All test cases passed.\n")

print("Anomalous Download Detected:\n")

for event in flagged:
    print(f"User      : {event['user']}")
    print(f"Action    : {event['action']}")
    print(f"File      : {event['file']}")
    print(f"Time      : {event['timestamp']}")
    print(f"IP Address: {event['ip']}")
    print("Reasons   :")
    for reason in event["reasons"]:
        print(f" - {reason}")

All test cases passed.

Anomalous Download Detected:

User      : alice
Action    : download
File      : customer_database.csv
Time      : 2026-03-05 02:15:00
IP Address: 185.220.101.7
Reasons   :
 - IP differs from user baseline
 - Outside business hours
